In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import random
import os
import logging
from tensorflow.keras.models import Sequential, clone_model, load_model
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import BinaryAccuracy, AUC
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, auc

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\keras\engine\training_arrays_v1.py:37: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 1.23.2)
  from scipy.sparse import issparse  # pylint: disable=g-import-not-at-top


In [3]:
# Set up logging
logging.basicConfig(filename='training.log', level=logging.INFO, format='%(asctime)s %(message)s')


In [5]:
# Set seed for reproducibility
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

In [6]:
# Load the dataset
data_x = pd.read_csv('X_cma.csv')
data_y = pd.read_csv('y_cma.csv')

In [16]:
# Remove the 'Unnamed: 0' column if it exists
if 'Unnamed: 0' in data_x.columns:
    data_x = data_x.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in data_y.columns:
    data_y = data_y.drop(columns=['Unnamed: 0'])

In [7]:
# Convert to numpy arrays
X = data_x.values
y = data_y['Target'].values

In [8]:
# Define sliding window parameters
window_size = 600
test_gap = 180
step_size = 1

In [9]:
# Determine the start point to process only the last 500 windows
start_point = len(X) - (500 * step_size) - window_size - test_gap

In [10]:
# Lists to store results
all_test_predictions = []
all_y_pred_results = []
all_y_true_results = []
all_confusion_metrics = []
all_classification_reports = []

In [11]:
# Define model creation function
def create_model(hu=20, lookback=10, features=1):
    tf.keras.backend.clear_session()
    model = Sequential()
    model.add(LSTM(units=hu, input_shape=(lookback, features), activation='relu', return_sequences=False, name='LSTM1'))
    model.add(Dropout(0.4, name='Dropout1'))
    model.add(Dense(units=1, activation='sigmoid', name='Output'))
    opt = Adam(learning_rate=0.001, epsilon=1e-08, decay=0.0)
    model.compile(optimizer=opt, loss=BinaryCrossentropy(), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
    return model

In [12]:
# Initialize and save the initial model
initial_model = create_model(hu=20, lookback=10, features=X.shape[1])
initial_model.compile(optimizer=Adam(learning_rate=0.001, epsilon=1e-08, decay=0.0), loss=BinaryCrossentropy(), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
initial_model.save('initial_model.h5')

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


In [15]:
# Save the weights of the initial model
initial_model.save_weights('current_model.weights.h5')

In [16]:
# EarlyStopping callback
early_stopping = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

In [17]:
# Checkpoint callback
checkpoint = ModelCheckpoint(filepath='model_checkpoint.weights.h5', save_best_only=True, save_weights_only=True, monitor='loss', mode='min')


In [24]:
# Train and evaluate the model with sliding window
for start in range(0, len(X) - window_size - test_gap, step_size):
    end = start + window_size
    X_train_window = X[start:end]
    y_train_window = y[start:end]
    
    # Define test indices
    test_start = end + test_gap
    test_end = test_start + 1
    if test_end > len(X):
        break
    X_test_window = X[test_start:test_end]
    y_test_window = y[test_start:test_end]

    # Sequence length for training
    seqlen = 10
    # Number of features
    numfeat = X_train_window.shape[1]

    # Generate sequence input for the training dataset
    train_generator = TimeseriesGenerator(X_train_window, y_train_window, length=seqlen, batch_size=32)

    # Load and clone the initial model for the first iteration, then the previously trained model for subsequent iterations
    model = create_model(hu=20, lookback=10, features=numfeat)
    model.load_weights('current_model.weights.h5')

    model_clone = clone_model(model)
    model_clone.set_weights(model.get_weights())

    model_transfer = Sequential(model_clone.layers[:-1])
    model_transfer.add(Dense(1, activation='sigmoid'))

    # Freeze the cloned layers initially
    for layer in model_transfer.layers[:-1]:
        layer.trainable = False

    model_transfer.compile(loss=BinaryCrossentropy(), optimizer=SGD(learning_rate=1e-3), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
    model_transfer.fit(train_generator, epochs=4, verbose=1, class_weight=dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_window), y=y_train_window))), shuffle=False, callbacks=[early_stopping])

    # Unfreeze the cloned layers for further training
    for layer in model_transfer.layers[:-1]:
        layer.trainable = True

    model_transfer.compile(loss=BinaryCrossentropy(), optimizer=SGD(learning_rate=1e-3), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
    model_transfer.fit(train_generator, epochs=70, verbose=1, class_weight=dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_window), y=y_train_window))), shuffle=False, callbacks=[early_stopping])

    # Save the best model during training for the current window
    model_transfer.save('current_model.h5')
    model_transfer.save_weights('current_model.weights.h5')

    # Predict on the single test sample directly
    test_input = np.expand_dims(X_test_window, axis=0)
    prediction = model_transfer.predict(test_input)
    print(f'Test Prediction: {prediction[0][0]:.2f}')

    # Evaluate with the training set
    y_true_train = y_train_window[-(len(X_train_window)-seqlen):]
    y_pred_train_probs = model_transfer.predict(train_generator)
    y_pred_train = (y_pred_train_probs > 0.5).astype(int)

    # Calculate evaluation metrics
    train_accuracy = accuracy_score(y_true_train, y_pred_train)
    train_precision = precision_score(y_true_train, y_pred_train)
    train_recall = recall_score(y_true_train, y_pred_train)
    train_f1 = f1_score(y_true_train, y_pred_train)
    train_roc_auc = roc_auc_score(y_true_train, y_pred_train_probs)

    #print(f'Training Accuracy: {train_accuracy:.2f}')
    #print(f'Training Precision: {train_precision:.2f}')
    #print(f'Training Recall: {train_recall:.2f}')
    #print(f'Training F1 Score: {train_f1:.2f}')
    #print(f'Training ROC AUC: {train_roc_auc:.2f}')

    # Store results
    all_test_predictions.append(prediction[0][0])
    all_y_pred_results.extend(y_pred_train.tolist())
    all_y_true_results.extend(y_true_train.tolist())
    all_confusion_metrics.extend(confusion_matrix(y_true_train, y_pred_train).ravel().tolist())
    all_classification_reports.append(classification_report(y_true_train, y_pred_train, output_dict=True))

    # Evaluate with the test set
    y_true_test = y_test_window
    y_pred_test_probs = prediction  # Use prediction directly as y_pred_probs

    # Binary classification on test set
    y_pred_test_cm = (y_pred_test_probs > 0.5).astype(int)

    # Print classification report for test set
    #print("Classification Report for Test Set")
    #print(classification_report(y_true_test, y_pred_test_cm))

    # Determine confusion matrix for test set
    cm_test = confusion_matrix(y_true_test, y_pred_test_cm)
    if cm_test.shape == (1, 1):
        tn, fp, fn, tp = cm_test[0, 0], 0, 0, 0
    elif cm_test.shape == (2, 2):
        tn, fp, fn, tp = cm_test.ravel()
    else:
        raise ValueError("Unexpected confusion matrix shape")

    # Store results for the test set
    all_confusion_metrics.extend([tn, fp, fn, tp])
    all_classification_reports.append(classification_report(y_true_test, y_pred_test_cm, output_dict=True))

    # Determine AUC for the test set
    fpr_test, tpr_test, thresholds_test = roc_curve(y_true_test, y_pred_test_probs)
    roc_auc_test = auc(fpr_test, tpr_test)

Epoch 1/4


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


UnknownError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
ZMQError: Too many open files
Traceback (most recent call last):

  File "C:\Users\tonfo\AppData\Roaming\Python\Python311\site-packages\ipykernel\iostream.py", line 131, in _event_pipe
    event_pipe = self._local.event_pipe
                 ^^^^^^^^^^^^^^^^^^^^^^

AttributeError: '_thread._local' object has no attribute 'event_pipe'


During handling of the above exception, another exception occurred:


Traceback (most recent call last):

  File "c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\ops\script_ops.py", line 270, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\autograph\impl\api.py", line 643, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

  File "c:\Users\tonfo\AppData\Loc

: 

In [18]:
# Train and evaluate the model with sliding window
for start in range(start_point, len(X) - window_size - test_gap, step_size):
    try:
        end = start + window_size
        X_train_window = X[start:end]
        y_train_window = y[start:end]
        
        # Define test indices
        test_start = end + test_gap
        test_end = test_start + 1
        if test_end > len(X):
            break
        X_test_window = X[test_start:test_end]
        y_test_window = y[test_start:test_end]

        logging.info(f'Training window: {start} to {end}')
        logging.info(f'Test window: {test_start} to {test_end}')
        logging.info(f'X_train_window shape: {X_train_window.shape}, y_train_window shape: {y_train_window.shape}')
        logging.info(f'X_test_window shape: {X_test_window.shape}, y_test_window shape: {y_test_window.shape}')

        # Sequence length for training
        seqlen = 10
        # Number of features
        numfeat = X_train_window.shape[1]

        # Generate sequence input for the training dataset
        train_generator = TimeseriesGenerator(X_train_window, y_train_window, length=seqlen, batch_size=32)
        logging.info(f'Train generator length: {len(train_generator)}')

        # Load and clone the initial model for the first iteration, then the previously trained model for subsequent iterations
        model = create_model(hu=20, lookback=10, features=numfeat)
        model.load_weights('current_model.weights.h5')

        model_clone = clone_model(model)
        model_clone.set_weights(model.get_weights())

        model_transfer = Sequential(model_clone.layers[:-1])
        model_transfer.add(Dense(1, activation='sigmoid'))

        # Freeze the cloned layers initially
        for layer in model_transfer.layers[:-1]:
            layer.trainable = False

        model_transfer.compile(loss=BinaryCrossentropy(), optimizer=SGD(learning_rate=1e-3), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
        model_transfer.fit(train_generator, epochs=4, verbose=1, class_weight=dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_window), y=y_train_window))), shuffle=False, callbacks=[early_stopping, checkpoint])

        # Unfreeze the cloned layers for further training
        for layer in model_transfer.layers[:-1]:
            layer.trainable = True

        model_transfer.compile(loss=BinaryCrossentropy(), optimizer=SGD(learning_rate=1e-3), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
        model_transfer.fit(train_generator, epochs=70, verbose=1, class_weight=dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_window), y=y_train_window))), shuffle=False, callbacks=[early_stopping, checkpoint])

        # Save the best model during training for the current window
        model_transfer.save('current_model.h5')
        model_transfer.save_weights('current_model.weights.h5')

        # Predict on the single test sample directly
        test_input = np.expand_dims(X_test_window, axis=0)
        prediction = model_transfer.predict(test_input)
        logging.info(f'Test Prediction: {prediction[0][0]:.2f}')

        # Evaluate with the training set
        y_true_train = y_train_window[-(len(X_train_window)-seqlen):]
        y_pred_train_probs = model_transfer.predict(train_generator)
        y_pred_train = (y_pred_train_probs > 0.5).astype(int)

        # Calculate evaluation metrics
        train_accuracy = accuracy_score(y_true_train, y_pred_train)
        train_precision = precision_score(y_true_train, y_pred_train)
        train_recall = recall_score(y_true_train, y_pred_train)
        train_f1 = f1_score(y_true_train, y_pred_train)
        train_roc_auc = roc_auc_score(y_true_train, y_pred_train_probs)

        logging.info(f'Training Accuracy: {train_accuracy:.2f}')
        logging.info(f'Training Precision: {train_precision:.2f}')
        logging.info(f'Training Recall: {train_recall:.2f}')
        logging.info(f'Training F1 Score: {train_f1:.2f}')
        logging.info(f'Training ROC AUC: {train_roc_auc:.2f}')

        # Store results
        all_test_predictions.append(prediction[0][0])
        all_y_pred_results.extend(y_pred_train.tolist())
        all_y_true_results.extend(y_true_train.tolist())
        all_confusion_metrics.extend(confusion_matrix(y_true_train, y_pred_train).ravel().tolist())

        # Evaluate with the test set
        y_true_test = y_test_window
        y_pred_test_probs = prediction  # Use prediction directly as y_pred_probs

        # Binary classification on test set
        y_pred_test_cm = (y_pred_test_probs > 0.5).astype(int)

        # Print classification report for test set
        logging.info("Classification Report for Test Set")
        logging.info(classification_report(y_true_test, y_pred_test_cm))

        # Determine confusion matrix for test set
        cm_test = confusion_matrix(y_true_test, y_pred_test_cm)
        if cm_test.shape == (1, 1):
            tn, fp, fn, tp = cm_test[0, 0], 0, 0, 0
        elif cm_test.shape == (2, 2):
            tn, fp, fn, tp = cm_test.ravel()
        else:
            raise ValueError("Unexpected confusion matrix shape")

        # Store results for the test set
        all_confusion_metrics.extend([tn, fp, fn, tp])

        # Determine AUC for the test set
        fpr_test, tpr_test, thresholds_test = roc_curve(y_true_test, y_pred_test_probs)
        roc_auc_test = auc(fpr_test, tpr_test)

    except Exception as e:
        logging.error(f'Error at window {start} to {end}: {e}')
        break

Epoch 1/4


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6270 - auc_1: 0.5697 - loss: 6.1892    
Epoch 2/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6459 - auc_1: 0.6107 - loss: 3.0381      
Epoch 3/4


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6618 - auc_1: 0.6266 - loss: 1.9148     
Epoch 4/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6928 - auc_1: 0.6720 - loss: 1.1484     
Epoch 1/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7002 - auc_2: 0.7607 - loss: 1.0986       
Epoch 2/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4595 - auc_2: 0.4699 - loss: 0.6924     
Epoch 3/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4595 - auc_2: 0.4699 - loss: 0.6924     
Epoch 4/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4595 - auc_2: 0.4824 - loss: 0.6924     
Epoch 5/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4595 - auc_2: 0.4824 - loss: 0.6924     
Epoch 6/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4595 - auc_2: 0.4532 - loss: 0.6924     
Epoch 7/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4595 - auc_2: 0.4532 - loss: 0.6924     
Epoch 8/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, 

... (2,960 outputs left out)

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5645 - auc_1: 0.3250 - loss: 0.6588           
Epoch 2/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7447 - auc_1: 0.3250 - loss: 0.6588     
Epoch 3/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7447 - auc_1: 0.3250 - loss: 0.6588     
Epoch 4/4


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7447 - auc_1: 0.3250 - loss: 0.6588     
Epoch 1/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7447 - auc_2: 0.3250 - loss: 0.6588       
Epoch 2/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7447 - auc_2: 0.3250 - loss: 0.6588     
Epoch 3/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7447 - auc_2: 0.3250 - loss: 0.6588     
Epoch 4/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7471 - auc_2: 0.3250 - loss: 0.6588     
Epoch 5/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7471 - auc_2: 0.3250 - loss: 0.6588     
Epoch 6/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7471 - auc_2: 0.3250 - loss: 0.6588     
Epoch 7/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7471 - auc_2: 0.3250 - loss: 0.6588     
Epoch 8/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7359 - auc_2: 0.3250 - loss: 0.6588     
Epoch 9/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:395: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_ranking.py:1174: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(


In [19]:
# Save the final model
model_transfer.save('final_model.h5')


In [22]:
# Ensure all arrays are of the same length before saving results
min_length = min(len(all_test_predictions), len(all_y_pred_results), len(all_y_true_results), len(all_confusion_metrics) // 4)

all_test_predictions = all_test_predictions[:min_length]
all_y_pred_results = all_y_pred_results[:min_length]
all_y_true_results = all_y_true_results[:min_length]
all_confusion_metrics = all_confusion_metrics[:min_length * 4]

# Reshape confusion metrics to ensure each row corresponds to one prediction
confusion_metrics_reshaped = np.array(all_confusion_metrics).reshape(-1, 4)

# Save the results to files for further analysis
results = {
    "test_predictions": all_test_predictions,
    "y_pred_results": all_y_pred_results,
    "y_true_results": all_y_true_results
}

# Create a DataFrame for the primary results
results_df = pd.DataFrame(results)
results_df.to_csv('model_results.csv', index=False)

# Create a DataFrame for the confusion metrics
confusion_metrics_df = pd.DataFrame(confusion_metrics_reshaped, columns=['tn', 'fp', 'fn', 'tp'])
confusion_metrics_df.to_csv('confusion_metrics.csv', index=False)


In [ ]:
# Save the final model
model_transfer.save('final_model.h5')

NameError: name 'model_transfer' is not defined

In [ ]:
# Save the results to files for further analysis
results = {
    "test_predictions": all_test_predictions,
    "y_pred_results": all_y_pred_results,
    "y_true_results": all_y_true_results,
    "confusion_metrics": all_confusion_metrics,
    "classification_reports": all_classification_reports
}

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv('CMA_results.csv', index=False)